# Task 3.1 — OTP Daily Report (DGCA Regulatory Table)

Per `airline + route + departure_date`:
- `total_flights`, `on_time_count`, `otp_14_pct` (delay < 14 min), `otp_0_pct`
- `avg_delay_minutes`, `p95_delay_minutes`, `cancellation_count`
- Compare `otp_14_pct` vs airline's `otp_target_pct` from SLA → `compliance_status`
- Partition by `departure_date`; Z-ORDER by `airline_code` for fast regulatory query
- **SLA**: ready by 06:00 AM — submitted to DGCA monthly in aggregated form


In [ ]:
# ── Imports & Configuration ────────────────────────────────────────
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import *
from delta.tables import DeltaTable

BRONZE_DIR = "/FileStore/delta_lake/bronze"
SILVER_DIR = "/FileStore/delta_lake/silver"
GOLD_DIR = "/FileStore/delta_lake/gold"

try:
    spark
except NameError:
    spark = (SparkSession.builder
        .appName("Task_3_1_OTP_Daily_Report")
        .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.1.0")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog",
                "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .getOrCreate())


## Step 1 — Read Silver Flight Operations & SLA

In [ ]:
# ── Read required tables ──────────────────────────────────────────
flight_ops_df = spark.read.format("delta").load(f"{SILVER_DIR}/flight_operations")
sla_df = spark.read.format("delta").load(f"{BRONZE_DIR}/airline_sla")

# Prepare flight data with route
flights_df = (flight_ops_df
    .withColumn("departure_date", F.to_date("scheduled_departure"))
    .withColumn("route", F.concat_ws("-",
        F.col("origin_iata"), F.col("destination_iata")))
)

print(f"Flight operations: {flights_df.count()}")
print(f"SLA contracts: {sla_df.count()}")
flights_df.select("flight_id", "airline_code", "route", "departure_date",
                  "delay_minutes", "status").show(5, truncate=False)


## Step 2 — Aggregate OTP Metrics per Airline + Route + Date

In [ ]:
# ── OTP Aggregation ──────────────────────────────────────────────
otp_df = (flights_df
    .groupBy("airline_code", "airline_name", "route",
             "origin_iata", "destination_iata", "departure_date")
    .agg(
        F.count("*").alias("total_flights"),

        # On-time counts
        F.sum(F.when(F.col("delay_minutes") == 0, 1).otherwise(0))
            .alias("on_time_exact_count"),
        F.sum(F.when(F.col("delay_minutes") <= 14, 1).otherwise(0))
            .alias("on_time_14_count"),

        # Delay stats
        F.round(F.avg("delay_minutes"), 1).alias("avg_delay_minutes"),
        F.expr("percentile_approx(delay_minutes, 0.95)")
            .alias("p95_delay_minutes"),
        F.max("delay_minutes").alias("max_delay_minutes"),

        # Cancellations
        F.sum(F.when(F.col("status") == "CNX", 1).otherwise(0))
            .alias("cancellation_count")
    )
)

# Compute OTP percentages
otp_df = (otp_df
    .withColumn("otp_0_pct",
        F.round(F.col("on_time_exact_count") / F.col("total_flights") * 100, 1))
    .withColumn("otp_14_pct",
        F.round(F.col("on_time_14_count") / F.col("total_flights") * 100, 1))
)

print(f"OTP report rows: {otp_df.count()}")
otp_df.show(5, truncate=False)


## Step 3 — Compare vs SLA Targets → Compliance Status

In [ ]:
# ── Join SLA targets and compute compliance ──────────────────────
otp_with_sla = otp_df.join(
    sla_df.select("airline_code", "otp_target_pct"),
    on="airline_code",
    how="left"
)

otp_with_sla = otp_with_sla.withColumn(
    "compliance_status",
    F.when(F.col("otp_14_pct") >= F.col("otp_target_pct"), "MEETING_SLA")
     .when(F.col("otp_14_pct") >= F.col("otp_target_pct") - 5, "BELOW_SLA")
     .otherwise("CRITICAL")
)

otp_with_sla = otp_with_sla.withColumn("report_generated_ts", F.current_timestamp())

print("\nCompliance Status Distribution:")
otp_with_sla.groupBy("compliance_status").count().show()

print("\nSample OTP Report:")
otp_with_sla.select(
    "airline_code", "route", "departure_date", "total_flights",
    "otp_14_pct", "otp_target_pct", "avg_delay_minutes", "compliance_status"
).orderBy("airline_code", "route").show(10, truncate=False)


## Step 4 — Write to Gold Delta

In [ ]:
# ── Write gold.otp_daily_report ───────────────────────────────────
gold_otp_path = f"{GOLD_DIR}/otp_daily_report"

(otp_with_sla.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("departure_date")
    .save(gold_otp_path))

print(f"Written to {gold_otp_path}")
result_df = spark.read.format("delta").load(gold_otp_path)
print(f"Gold otp_daily_report total: {result_df.count()}")

# Summary by airline
print("\nOTP Summary by Airline:")
result_df.groupBy("airline_code", "compliance_status").agg(
    F.sum("total_flights").alias("flights"),
    F.round(F.avg("otp_14_pct"), 1).alias("avg_otp_14_pct"),
    F.round(F.avg("avg_delay_minutes"), 1).alias("avg_delay_mins")
).orderBy("airline_code").show(20, truncate=False)
